<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/17_regression_crossval/17_1_SLR/17_1_2_SLR_Significance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple Linear Regression: Statistical Significance (Did We Just Get Lucky?)

Author: Brad Sheese

---

## What This Notebook Is About

In Notebook 17_1_1 we fit a line through 333 Palmer Penguins and got a slope of **50.15 g/mm** with an $R^2$ of **0.76**. Then we dropped the statsmodels `.summary()` table on you and ignored 90% of it.

This notebook is about the question hiding behind every regression result:

> **Did we actually discover something real — or did we just get lucky with which 333 penguins happened to land in our dataset?**

Statistics answers this with three tools from the summary table:

| Column | Symbol | What it's for |
|---|---|---|
| `std err` | $\text{SE}(\hat{m})$ | How much the slope wiggles across different samples |
| `P>\|t\|` | $p$ | Odds of seeing a slope this big by pure luck |
| `[0.025  0.975]` | 95% CI | A range of slopes consistent with the data |

We'll build each from the ground up using **simulation**, then match what we simulate against what statsmodels prints.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

sns.set_style('whitegrid')
rng = np.random.default_rng(seed=42)

penguins = sns.load_dataset('penguins').dropna().reset_index(drop=True)
x = penguins['flipper_length_mm']
y = penguins['body_mass_g']

X_sm = sm.add_constant(x)
model_sm = sm.OLS(y, X_sm).fit()

print(model_sm.summary())

Focus on the row labeled `flipper_length_mm`:

```
                         coef    std err         t      P>|t|    [0.025    0.975]
flipper_length_mm     50.1533     1.540    32.562      0.000    47.124    53.183
```

We covered the first column (`coef`) in 17_1_1. This notebook is about the other five.

---

## Section 1: The Null Hypothesis

What if flipper length tells us *nothing* about body mass? What if the upward trend is just a cosmic accident of which 333 penguins waddled into our sample?

Coincidences happen. Flip a fair coin 10 times and sometimes you get 8 heads. If flipper length and body mass were truly unrelated, *sometimes* the longer-flippered penguins would happen to be heavier purely by chance.

This is the **null hypothesis**, $H_0$:

> $H_0$: **The true slope is zero.** Flipper length and body mass have no real relationship.

Against the **alternative** $H_A$:

> $H_A$: **The true slope is not zero.** There is a real relationship.

Our job is to pile up enough evidence that $H_0$ becomes absurd. How do we make "could plausibly have come from $H_0$" precise? That's what the next three sections are for.

---

## Section 2: The Standard Error — How Much Does the Slope Wiggle?

> If I collected 333 *different* penguins tomorrow, how different would my slope be?

Our 50.15 came from one sample. A different sample would give a slightly different slope. Statsmodels summarizes this wiggle as **`std err` = 1.540** — the estimated standard deviation of the slope across hypothetical repeat samples.

How could statsmodels know that from a single sample? The classical formula makes assumptions (which we'll check in 17_1_3). For now, we'll skip the formula and **simulate** the wiggle directly using the **bootstrap**: resample our 333 penguins *with replacement*, refit, and record the slope — 2000 times.

In [ ]:
N = len(penguins)
n_iter = 2000
bootstrap_slopes = np.empty(n_iter)

# For speed, use the closed-form SLR slope from 17_0_5.
# np.cov(..., bias=True) uses the population formula (divide by n), matching the OLS derivation.
x_arr = x.to_numpy()
y_arr = y.to_numpy()

for i in range(n_iter):
    idx = rng.integers(0, N, size=N)
    xb, yb = x_arr[idx], y_arr[idx]
    bootstrap_slopes[i] = np.cov(xb, yb, bias=True)[0, 1] / np.var(xb)

bootstrap_mean = bootstrap_slopes.mean()
bootstrap_std  = bootstrap_slopes.std(ddof=1)

print(f'Across {n_iter} bootstrap samples:')
print(f'  Mean slope:                   {bootstrap_mean:.4f} g/mm')
print(f'  Std dev of slopes (our SE):   {bootstrap_std:.4f} g/mm')
print(f'  Statsmodels std err from summary: {model_sm.bse["flipper_length_mm"]:.4f} g/mm')

Our bootstrap standard deviation (~1.49) is essentially identical to the `std err` from the summary table (~1.54).

> **That's what `std err` means.** It's the standard deviation of the slope across hypothetical repeat samples. Statsmodels uses a formula; we used brute-force simulation. They agree.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.hist(bootstrap_slopes, bins=50, color='steelblue', alpha=0.75, edgecolor='white')
ax.axvline(bootstrap_mean, color='darkorange', linewidth=2,
           label=f'Bootstrap mean = {bootstrap_mean:.2f}')
ax.axvline(0, color='red', linestyle='--', linewidth=2,
           label='Null hypothesis: slope = 0')
ax.set_xlabel('Slope (g/mm) from a resampled dataset')
ax.set_ylabel('Count')
ax.set_title(f'Distribution of the slope across {n_iter} bootstrap samples')
ax.set_xlim(-5, 60)
ax.legend()
plt.show()

Two things to notice:

1. **It's roughly bell-shaped and tight**, centered around 50, with most mass between about 47 and 53. Not a single resample gave a slope near zero.
2. **The red dashed line (null slope = 0) is miles away.**

That visual gap is already a strong clue that $H_0$ is in trouble. Let's also connect this bootstrap SE to the next column in the summary table: the **t-statistic**.

### The t-statistic: bridging SE and p-value

The t-statistic is simply the slope divided by its standard error:

$$t = \frac{\hat{\beta}_1}{\text{SE}(\hat{\beta}_1)}$$

It measures "how many standard errors our slope is from zero." A large |t| means the slope is many SEs away from zero — strong evidence against $H_0$.

In [ ]:
slope = model_sm.params['flipper_length_mm']
se = model_sm.bse['flipper_length_mm']
t_stat = slope / se

print(f'Slope:               {slope:.4f}')
print(f'SE:                   {se:.4f}')
print(f't = slope / SE:      {t_stat:.3f}')
print(f'Statsmodels t-column: {model_sm.tvalues["flipper_length_mm"]:.3f}')

The slope (50.15) is about 32.6 standard errors away from zero — an enormous distance on the t-distribution, which is why the p-value is essentially zero.

Now we have all the ingredients for the next section: the p-value asks, *"if $H_0$ were true, what's the chance of seeing a t-statistic this extreme?"*

---

## Section 3: The P-Value — Simulating the Null World

The bootstrap told us how much the slope wiggles *in a world where our relationship is real*. The p-value asks the opposite: **in a world where $H_0$ is true, how rare is a slope like ours?**

The key difference in method:
- **Bootstrap** resamples *pairs* of (x, y) with replacement, preserving the x–y relationship. This estimates the sampling distribution under the *real* model.
- **Permutation test** (below) shuffles y randomly, *breaking* the x–y relationship. This simulates the *null* distribution — what slopes would look like if $H_0$ were true.

Same resampling idea, different goal.

In [ ]:
n_iter = 5000
null_slopes = np.empty(n_iter)

for i in range(n_iter):
    y_shuffled = rng.permutation(y_arr)        # destroy any real relationship
    null_slopes[i] = np.cov(x_arr, y_shuffled, bias=True)[0, 1] / np.var(x_arr)

observed_slope = model_sm.params['flipper_length_mm']

# Two-sided empirical p-value: fraction of null slopes at least as extreme as ours.
p_empirical = np.mean(np.abs(null_slopes) >= np.abs(observed_slope))
# Report resolution limit honestly.
p_display = max(p_empirical, 1 / n_iter)

print(f'Observed slope:                    {observed_slope:.4f} g/mm')
print(f'Null distribution mean:            {null_slopes.mean():.4f} g/mm  (should be ~0)')
print(f'Null distribution std dev:         {null_slopes.std(ddof=1):.4f} g/mm')
print(f'Empirical p-value (of {n_iter} shuffles): {p_display:.4f}  (< 1/{n_iter} = {1/n_iter:.4f})')
print(f'Statsmodels p-value:               {model_sm.pvalues["flipper_length_mm"]:.3g}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.hist(null_slopes, bins=50, color='steelblue', alpha=0.75, edgecolor='white',
        label='Null-world slopes (y shuffled)')
ax.axvline(observed_slope, color='red', linewidth=2.5,
           label=f'Our observed slope = {observed_slope:.2f}')
ax.axvline(-observed_slope, color='red', linewidth=2.5, linestyle=':')
ax.set_xlabel('Slope (g/mm) under the null hypothesis')
ax.set_ylabel('Count')
ax.set_title(f'Null distribution: {n_iter} shuffles. Our slope is nowhere on this plot.')
ax.set_xlim(-55, 55)
ax.legend()
plt.show()

The blue histogram is **every slope the null world could plausibly produce** — centered on zero, stretching from about −5 to +5. Our slope of 50.15 is so far to the right that it doesn't fit in the frame. Out of 5000 shuffles, not a single one produced a slope anywhere near as extreme.

> **If flipper length had no real relationship with body mass, we would essentially never see a slope of 50 in a sample of 333 penguins. So we reject $H_0$.**

That is what "statistically significant" means. It is a claim about one very specific thing: *the data we observed is extremely unlikely if the null hypothesis were true.*

### The 0.05 convention (and its discontents)

By convention, scientists "reject $H_0$" when $p < 0.05$. This is a **convention**, not a law of nature.

1. **It is arbitrary.** A p-value of 0.049 and 0.051 describe essentially identical evidence, but one "passes" and the other "fails."
2. **A small p-value is not a big effect.** With enough data, a microscopic slope becomes "statistically significant." Always look at the *size* of the coefficient, not just the p-value.

Our p-value of essentially zero, combined with a slope of 50 g/mm and $R^2 = 0.76$, is a rare luxury: the effect is both highly significant *and* large. Don't expect that combination every time.

### A small p-value does not mean a big effect

With a large enough sample, *any* nonzero slope — no matter how tiny — becomes statistically significant. Let's see this in action.

In [ ]:
# Synthetic data: tiny effect, huge sample size.
rng_big = np.random.default_rng(seed=42)
n_big = 10000
x_big = rng_big.uniform(0, 100, size=n_big)
y_big = 50 + 0.01 * x_big + rng_big.normal(0, 5, size=n_big)
model_big = sm.OLS(y_big, sm.add_constant(x_big)).fit()
print(f'Slope: {model_big.params[1]:.4f}')
print(f'SE:    {model_big.bse[1]:.4f}')
print(f'p:     {model_big.pvalues[1]:.4g}')
print(f'R^2:   {model_big.rsquared:.4f}')

The slope is 0.01 — increasing $x$ by 100 units changes $y$ by only 1 unit. Yet $p < 0.001$ and the CI won't include zero. The effect is real but tiny. With enough data, *any* nonzero effect becomes statistically significant. Always check the *size* of the coefficient, not just the p-value.

---

## Section 4: The 95% Confidence Interval

We've rejected "the slope is zero." But what *is* the slope?

Our point estimate is 50.15 g/mm. The `std err` told us the wiggle is about 1.54. The **95% confidence interval** translates that wiggle into a range of plausible values. Statsmodels gives `[47.124, 53.183]`.

We can get this two ways: via the bootstrap (empirical percentiles) or via the formula $\hat{\beta}_1 \pm t^* \cdot \text{SE}$.

In [ ]:
# Method 1: Bootstrap percentile interval.
ci_low, ci_high = np.percentile(bootstrap_slopes, [2.5, 97.5])

# Method 2: Formula-based CI using t-distribution.
from scipy import stats as scipy_stats
t_crit = scipy_stats.t.ppf(0.975, df=model_sm.df_resid)  # 2.5% tail, two-sided
ci_formula_low  = observed_slope - t_crit * se
ci_formula_high = observed_slope + t_crit * se

sm_ci_low, sm_ci_high = model_sm.conf_int().loc['flipper_length_mm']

print(f'Bootstrap percentile CI:  [{ci_low:.3f}, {ci_high:.3f}]')
print(f'Formula CI (t*SE):        [{ci_formula_low:.3f}, {ci_formula_high:.3f}]')
print(f'Statsmodels CI:           [{sm_ci_low:.3f}, {sm_ci_high:.3f}]')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.hist(bootstrap_slopes, bins=50, color='steelblue', alpha=0.6, edgecolor='white')
ax.axvspan(ci_low, ci_high, alpha=0.2, color='darkorange',
           label=f'Middle 95%: [{ci_low:.2f}, {ci_high:.2f}]')
ax.axvline(observed_slope, color='red', linewidth=2,
           label=f'Point estimate = {observed_slope:.2f}')
ax.set_xlabel('Slope (g/mm)')
ax.set_ylabel('Count')
ax.set_title('95% CI = middle 95% of bootstrap distribution')
ax.legend()
plt.show()

All three methods agree: our data is consistent with a true slope roughly between 47 and 53 grams per millimeter.

The percentile bootstrap is the simplest method, but it works best when the bootstrap distribution is roughly symmetric (as it is here). For skewed distributions, more advanced methods (BCa, bootstrap-t) exist.

### What a 95% CI actually means

A 95% confidence interval does **not** mean "there's a 95% chance the true slope is between 47.1 and 53.2." The true slope is a fixed unknown number — it's either in that interval or it isn't.

The correct interpretation: *if we repeated this experiment many times — each time sampling 333 penguins and computing a 95% CI — then about 95% of those intervals would contain the true slope.*

Let's demonstrate this with a simulation.

In [ ]:
# Simulate 1000 experiments: compute a bootstrap CI for each and check coverage.
# We treat our observed slope as the "true" slope for this demonstration.
true_slope = observed_slope
n_sims = 1000
contains = np.empty(n_sims, dtype=bool)

for sim in range(n_sims):
    # Bootstrap resample
    idx = rng.integers(0, N, size=N)
    xb, yb = x_arr[idx], y_arr[idx]
    b_slopes = np.empty(200)
    for j in range(200):
        idx2 = rng.integers(0, N, size=N)
        b_slopes[j] = np.cov(xb[idx2], yb[idx2], bias=True)[0, 1] / np.var(xb[idx2])
    lo, hi = np.percentile(b_slopes, [2.5, 97.5])
    contains[sim] = (lo <= true_slope <= hi)

coverage = contains.mean()
print(f'Out of {n_sims} simulated experiments:')
print(f'  {contains.sum()} CIs contained the true slope.')
print(f'  Coverage rate: {coverage:.1%}')

Close to 95% — the definition in action. Each individual CI either contains the true slope or it doesn't, but across many experiments, the procedure captures it about 95% of the time.

---

## Section 5: How Sample Size Affects Everything

All of the results above — the SE, p-value, and CI width — depend heavily on sample size. Let's see what happens if we had collected only 30 penguins instead of 333.

In [ ]:
# Subsample to 30 penguins and compare results.
small = penguins.sample(30, random_state=42)
X_small = sm.add_constant(small['flipper_length_mm'])
model_small = sm.OLS(small['body_mass_g'], X_small).fit()

full_se = model_sm.bse['flipper_length_mm']
small_se = model_small.bse['flipper_length_mm']
full_ci = model_sm.conf_int().loc['flipper_length_mm']
small_ci = model_small.conf_int().loc['flipper_length_mm']

print(f'n=333: SE = {full_se:.3f}, CI width = {full_ci[1] - full_ci[0]:.2f}')
print(f'n=30:  SE = {small_se:.3f}, CI width = {small_ci[1] - small_ci[0]:.2f}')
print()
print('With 30 penguins, the SE roughly triples and the CI is much wider.')
print('The same relationship becomes harder to detect with less data.')

Sample size directly determines whether you can detect an effect. With $n=333$ the relationship was obvious; with $n=30$ it's buried in noise. This is why underpowered studies (too small a sample) are a major problem in science — they miss real effects because the SE is too large.

This also connects forward to the **bias–variance tradeoff** in 17_1_6: more training data reduces variance without increasing bias, which is why "get more data" is often the best fix for overfitting.

---

## Putting It All Together

Here's what each number in the summary table actually means:

| Column | Value | What it is | How we simulated it |
|---|---|---|---|
| `coef` | 50.1533 | The slope our sample gave us | — |
| `std err` | 1.540 | How much the slope wiggles across hypothetical samples | Bootstrap of pairs |
| `t` | 32.56 | How many SEs the slope is from zero ($t = \hat{\beta}_1 / \text{SE}$) | Computed directly |
| `P>\|t\|` | ~0 | Probability of seeing this slope if $H_0$ were true | Permutation test (shuffle y) |
| `[0.025  0.975]` | [47.124, 53.183] | Range of true slopes consistent with our data | Bootstrap percentiles / $t^* \cdot \text{SE}$ |

The workflow of statistical inference:

> **Fit a line. Compute its slope. Ask "could this plausibly have come from a world where the true slope is zero?" If the answer is no, report the slope with a confidence interval so your reader knows how precise your estimate is.**

### Statistical significance vs. practical significance

We rejected $H_0$ and bounded the slope between 47 and 53 g/mm. But *is a 50 g/mm increase meaningful for a penguin?* That's a biological question, not a statistical one. Statistical significance tells you an effect is real; practical significance tells you it matters. Always ask both.

---

**YOUR TURN (10 min):** Re-run the bootstrap with `n_iter = 100` instead of 2000. How does the SE estimate change? How stable is the 95% CI? Try a few different random seeds by changing the seed in `np.random.default_rng(seed=...)`.

---

### Where We're Going Next

All of the classical statsmodels output — the `std err`, the p-value, the CI — is computed from formulas that assume certain things about the data. The bootstrap matched those formulas for our penguins, suggesting the assumptions held. But what if they didn't?

In `17_1_3_SLR_LINE_Assumptions.ipynb` we'll meet the four LINE assumptions and learn how to check them with diagnostic plots.